## Google Colab setup

Set `REPO_URL`, `BRANCH`, `RUN_NAME` and `BASE` in the next cell. Open this notebook from the same GitHub branch. The setup mounts Drive, clones code into the runtime and installs dependencies; local execution skips it. Use a fresh runtime after pushing new code. All notebooks in one experiment use the same run name; choose a new name for a fresh experiment. Run preprocessing and feature engineering to completion before model notebooks. The repository README describes the dataset layout and execution order.


In [ ]:
# Configure this cell before running on Google Colab.
import os
import sys
import subprocess
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    REPO_URL = "https://github.com/YOUR_ACCOUNT/YOUR_REPOSITORY.git"
    BRANCH = "YOUR_BRANCH"
    RUN_NAME = "run_01"
    BASE = Path("/content/drive/MyDrive/BT4222")
    CHECKOUT = Path("/content/bt4222_checkout")
    drive.mount("/content/drive")
    if not CHECKOUT.exists():
        subprocess.run([
            "git", "clone", "--branch", BRANCH, "--single-branch",
            REPO_URL, str(CHECKOUT)
        ], check=True)
    active_branch = subprocess.check_output(
        ["git", "-C", str(CHECKOUT), "branch", "--show-current"], text=True
    ).strip()
    if active_branch != BRANCH:
        raise ValueError("Checkout branch differs from BRANCH. Use a fresh Colab runtime.")
    CODE_DIR = (CHECKOUT if (CHECKOUT / "utils/data_utils.py").is_file()
                else CHECKOUT / "bt4222-group12")
    if not (CODE_DIR / "utils/data_utils.py").is_file():
        raise FileNotFoundError("The selected branch does not contain the Group 12 project.")
    os.environ.update({
        "BT4222_BASE": str(BASE),
        "MIND_DATA_DIR": str(BASE / "data"),
        "BT4222_PROCESSED_DIR": str(BASE / "processed"),
        "BT4222_FEATURE_DIR": str(BASE / "features"),
        "BT4222_GLOVE_PATH": str(BASE / "data/glove/glove.6B.300d.txt"),
        "BT4222_OUTPUT_DIR": str(BASE / "runs" / RUN_NAME),
    })
    os.chdir(CODE_DIR)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-r", "requirements.txt"
    ], check=True)
    print("Code:", CODE_DIR)
    print("Features:", os.environ["BT4222_FEATURE_DIR"])
    print("Run outputs:", os.environ["BT4222_OUTPUT_DIR"])


# User-Based Collaborative Filtering Baseline

Build a binary user–article matrix from the configured training click/history log. Compare users by cosine similarity, exclude the query user and retain at most 200 neighbors. Score candidate $a$ by normalized neighbor support:

$$s(u,a)=\frac{\sum_{v\in N(u)}\mathrm{sim}(u,v)R_{va}}{\sum_{v\in N(u)}|\mathrm{sim}(u,v)|}.$$

## Inputs, evaluation and limits

Use processed full slates and the fixed training window. Zero neighbor strength, unknown users and unavailable articles receive the declared baseline zero score. No time decay, inverse-user-frequency weighting or validation grid search is applied. Scores and per-cohort metrics are exported independently of advanced UBCF.

Read availability and cold-cohort results together: missing collaborative evidence is different from evidence of low relevance. This model cannot infer the relevance of a new article from text alone.


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "utils" / "data_utils.py").is_file()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.data_utils import project_paths, load_preprocessed, load_table, save_table

if os.environ.get("BT4222_USE_DRIVE") == "1":
    from google.colab import drive

    drive.mount("/content/drive")

PATHS = project_paths(PROJECT_ROOT)
from utils.data_utils import source_fingerprint, validate_features, bind_feature_run

feature_id = validate_features(PATHS)
bind_feature_run(PATHS, feature_id)


In [ ]:
from utils.metrics import evaluate_scores, history_groups


In [ ]:
import json
import time
from functools import lru_cache
import numpy as np
import pandas as pd
from scipy import sparse

OUT_DIR = str(PATHS["outputs"] / "base_models" / "ubcf")
CACHE_DIR = os.path.join(OUT_DIR, "cache")
DATA_DIR = str(PATHS["raw"])
os.makedirs(CACHE_DIR, exist_ok=True)
SEED = 42
TRAIN_DAYS = ["2019-11-09", "2019-11-10", "2019-11-11", "2019-11-12", "2019-11-13"]
VAL_DAY = "2019-11-14"
TEST_DAY = "2019-11-15"
PHASES = {
    "stack_train": {"build_days": TRAIN_DAYS[:-1], "score_days": [TRAIN_DAYS[-1]]},
    "val": {"build_days": TRAIN_DAYS, "score_days": [VAL_DAY]},
    "test": {"build_days": TRAIN_DAYS, "score_days": [TEST_DAY]},
}
NEG_K = 4
KS = (5, 10)
WARM_MIN_HISTORY = 5
COLD_SEG = "Cold users (0 history)"
SPARSE_SEG = "Sparse users (1-4 history)"
WARM_SEG = "Warm users (>=5 history)"
USER_SEGS = (COLD_SEG, SPARSE_SEG, WARM_SEG)
OUTPUT_COLS = [
    "impression_id",
    "user_id",
    "news_id",
    "label",
    "score",
    "available",
    "user_cold",
    "item_cold",
]


def _has_parquet() -> bool:
    try:
        import pyarrow  # noqa: F401

        return True
    except ImportError:
        return False


def save_df(df: pd.DataFrame, path_no_ext: str) -> str:
    if _has_parquet():
        path = path_no_ext + ".parquet"
        df.to_parquet(path, index=False)
    else:
        path = path_no_ext + ".pkl"
        df.to_pickle(path)
    return path


def load_behaviors():
    beh, _ = load_preprocessed(PATHS)
    beh["history"] = beh["history"].str.split()
    beh["date"] = beh["time"].dt.strftime("%Y-%m-%d")
    return beh


def load_news():
    return pd.read_pickle(PATHS["features"] / "hybrid_news.pkl")


def load_candidates(beh):
    rows = beh[["impression_id", "impressions"]].copy()
    rows["impressions"] = rows["impressions"].str.split()
    rows = rows.explode("impressions", ignore_index=True)
    parts = rows["impressions"].str.rsplit("-", n=1, expand=True)
    cand = pd.DataFrame(
        {
            "impression_id": rows["impression_id"].to_numpy(),
            "news_id": parts[0].to_numpy(),
            "label": parts[1].astype(np.int32).to_numpy(),
        }
    )
    cand["candidate_position"] = cand.groupby("impression_id").cumcount()
    return cand.merge(
        beh[["impression_id", "user_id", "time", "date", "split"]],
        on="impression_id",
        how="left",
        validate="many_to_one",
    )


In [ ]:
def build_click_log(beh: pd.DataFrame, cand: pd.DataFrame, build_days) -> pd.DataFrame:
    """All clicks a base model may learn from when built on `build_days`:
       (a) pre-period reading histories of users active in the build window
           (no timestamp in MIND -> stamped at the start of the build window, i.e. oldest)
       (b) impression clicks inside the build window (with their real timestamps).
    Returns one row per (user_id, news_id) with the latest click time."""
    b = beh[beh["date"].isin(build_days)]
    hist = (
        b.sort_values("time")[["user_id", "history"]]
        .drop_duplicates("user_id", keep="last")
        .explode("history")
        .dropna()
        .rename(columns={"history": "news_id"})
    )
    hist["time"] = pd.Timestamp(min(build_days))
    clicks = cand[cand["date"].isin(build_days) & (cand["label"] == 1)][
        ["user_id", "news_id", "time"]
    ]
    log = pd.concat([hist[["user_id", "news_id", "time"]], clicks], ignore_index=True)
    log = log.sort_values("time").drop_duplicates(["user_id", "news_id"], keep="last")
    return log.reset_index(drop=True)


def scoring_histories(beh: pd.DataFrame, score_days) -> pd.DataFrame:
    """Reading history each user brings into the scoring window (earliest impression in the
    window, so no later information is used). Used to fold in users the build log lacks.
    """
    s = beh[beh["date"].isin(score_days)].sort_values("time")
    first = s.drop_duplicates("user_id", keep="first")[["user_id", "history"]]
    h = first.explode("history").dropna().rename(columns={"history": "news_id"})
    return h[["user_id", "news_id"]].reset_index(drop=True)


def user_query_log(
    build_log: pd.DataFrame, beh: pd.DataFrame, score_days, users
) -> pd.DataFrame:
    """Clicks known for each scoring user at scoring time = build-window clicks + history
    they arrive with. History items carry the oldest timestamp of the build window."""
    users = pd.Index(pd.unique(users))
    known = build_log[build_log["user_id"].isin(users)]
    h = scoring_histories(beh, score_days)
    h = h[h["user_id"].isin(users)].copy()
    h["time"] = build_log["time"].min()
    q = pd.concat([known, h], ignore_index=True)
    q = q.sort_values("time").drop_duplicates(["user_id", "news_id"], keep="last")
    return q.reset_index(drop=True)


def phase_candidates(cand: pd.DataFrame, phase: str) -> pd.DataFrame:
    return cand[cand["date"].isin(PHASES[phase]["score_days"])].reset_index(drop=True)


def add_cold_start_flags(
    c: pd.DataFrame, beh: pd.DataFrame, build_log: pd.DataFrame, score_days
) -> pd.DataFrame:
    """Click-history cold-start definitions:
    user_cold = user has 0 clicked articles known at scoring time
                (nothing in the build-window log AND an empty history on arrival)
    item_cold = article has 0 historical clicks in the build window
                (never in any build-window history nor clicked in a build-window impression)
    """
    q = user_query_log(build_log, beh, score_days, c["user_id"])
    warm_users = set(q["user_id"].unique())
    clicked_items = set(build_log["news_id"].unique())
    c = c.copy()
    c["user_cold"] = (~c["user_id"].isin(warm_users)).astype(np.int8)
    c["item_cold"] = (~c["news_id"].isin(clicked_items)).astype(np.int8)
    return c


@lru_cache(maxsize=1)
def user_history_groups() -> pd.Series:
    """impression_id -> reporting group, from the length of the impression's raw MIND `history`
    (same definition as the MLP's cold_user / sparse_user / warm_user slices).
    Evaluation only: unlike user_cold it ignores clicks in the build window, so a user
    in the cold group can still have build-window clicks for the base models to use."""
    n = load_behaviors().set_index("impression_id")["history"].str.len()
    return pd.Series(
        np.select([n == 0, n < WARM_MIN_HISTORY], [COLD_SEG, SPARSE_SEG], WARM_SEG),
        index=n.index,
    )


In [ ]:
def evaluate(df, score_col="score", ks=KS, seed=SEED, by_segment=True):
    return evaluate_scores(df, score_col, ks, seed, by_segment, user_history_groups())


In [ ]:
def save_scores(df: pd.DataFrame, model: str, phase: str, extra_cols=()) -> str:
    """Write a model's scores in the shared format (+ optional model-specific feature cols)."""
    missing = [c for c in OUTPUT_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"missing output columns: {missing}")
    cols = OUTPUT_COLS + [c for c in extra_cols if c not in OUTPUT_COLS]
    out = df[cols].copy()
    out["available"] = out["available"].astype(np.int8)
    os.makedirs(OUT_DIR, exist_ok=True)
    return save_df(out, os.path.join(OUT_DIR, f"scores_{model}_{phase}"))


In [ ]:
beh = load_behaviors()
cand = load_candidates(beh)
news = load_news()


## 1. Fixed baseline and evaluation

Use the training window and full validation/test slates. Unavailable recommendations receive zero scores. Hyperparameters are fixed; weighting and tuning are evaluated separately.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

log = build_click_log(beh, cand, TRAIN_DAYS)
users = pd.Index(log["user_id"].unique())
items = pd.Index(log["news_id"].unique())
matrix = sparse.csr_matrix(
    (
        np.ones(len(log)),
        (users.get_indexer(log["user_id"]), items.get_indexer(log["news_id"])),
    ),
    shape=(len(users), len(items)),
)
matrix.data[:] = 1.0
N_NEIGHBORS = 200

for phase in ["val", "test"]:
    frame = add_cold_start_flags(
        phase_candidates(cand, phase), beh, log, PHASES[phase]["score_days"]
    )
    frame["score"] = 0.0
    frame["available"] = 0
    for user_id, group in frame.groupby("user_id", sort=False):
        if user_id not in users:
            continue
        user_idx = users.get_loc(user_id)
        similarities = cosine_similarity(matrix[user_idx], matrix).ravel()
        similarities[user_idx] = 0.0
        neighbors = np.argsort(-similarities, kind="stable")[:N_NEIGHBORS]
        denominator = np.abs(similarities[neighbors]).sum()
        if denominator == 0:
            continue
        scores = (
            np.asarray(similarities[neighbors] @ matrix[neighbors]).ravel()
            / denominator
        )
        article_idx = items.get_indexer(group["news_id"])
        available = article_idx >= 0
        frame.loc[group.index[available], "score"] = scores[article_idx[available]]
        frame.loc[group.index[available], "available"] = 1
    save_scores(frame, "baseline_ubcf", phase)
    display(evaluate(frame))
